In [0]:
%sql
INSERT OVERWRITE bootcamp.gold.fact_propiedades (
  row_hash, zona_id, tipo_operacion_id, fecha_id,
  caracteristicas_id, orientacion_id,
  precio, expensas, precio_por_m2,
  metros_cuadrados_totales, metros_cuadrados_cubiertos,
  ambientes, url
)
SELECT
  MD5(CONCAT_WS('|', sp.url, CAST(sp.precio AS STRING))) AS row_hash,
  dz.zona_id,
  dt.tipo_operacion_id,
  CAST(DATE_FORMAT(sp.fecha_publicacion, 'yyyyMMdd') AS BIGINT) AS fecha_id,
  dc.caracteristicas_id,
  do.orientacion_id,
  sp.precio,
  sp.expensas,
  sp.precio_por_m2,
  sp.metros_cuadrados_totales,
  sp.metros_cuadrados_cubiertos,
  sp.ambientes,
  sp.url
FROM bootcamp.silver.propiedades sp
LEFT JOIN bootcamp.gold.dim_zona dz
  ON sp.partido = dz.partido AND sp.region = dz.region
LEFT JOIN bootcamp.gold.dim_tipo_operacion dt
  ON sp.tipo_operacion = dt.tipo_operacion AND sp.moneda = dt.moneda
LEFT JOIN bootcamp.gold.dim_caracteristicas dc
  ON COALESCE(sp.estado, 'sin especificar') = dc.estado
  AND COALESCE(sp.cochera, false) = dc.cochera
LEFT JOIN bootcamp.gold.dim_orientacion do
  ON COALESCE(sp.orientacion, 'sin especificar') = do.orientacion
LEFT JOIN bootcamp.gold.dim_tiempo dtm
  ON sp.fecha_publicacion = dtm.fecha;


In [0]:
-- Verificar: total debe coincidir con Silver
SELECT
  (SELECT COUNT(*) FROM bootcamp.silver.propiedades) AS silver_count,
  (SELECT COUNT(*) FROM bootcamp.gold.fact_propiedades) AS fact_count;